# SD on Colab GPU + Cloudflare Tunnel (fixed domain)

Setup once in Cloudflare Zero Trust → Tunnels → Public Hostname `sd-api.YOURDOMAIN.com` → `http://localhost:7860`

Paste your tunnel token below. V2 `.env`: `SD_WEBUI_URL=https://sd-api.YOURDOMAIN.com`

In [ ]:
# === CONFIG — edit these ===
CLOUDFLARE_TUNNEL_TOKEN = 'PASTE_YOUR_TUNNEL_TOKEN_HERE'
# Your fixed URL (must match Cloudflare Public Hostname):
PUBLIC_URL = 'https://sd-api.yourdomain.com'

In [ ]:
!nvidia-smi

In [ ]:
!git clone -q https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /content/sd-webui
!mkdir -p /content/sd-webui/models/Stable-diffusion
!wget -q -O /content/sd-webui/models/Stable-diffusion/v1-5-pruned-emaonly.safetensors \
  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors

In [ ]:
import subprocess, os, time
os.chdir('/content/sd-webui')
env = os.environ.copy()
env['COMMANDLINE_ARGS'] = '--api --listen --port 7860 --xformers'
subprocess.Popen(['python', 'launch.py'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Loading SD WebUI (~2-3 min)...')
time.sleep(150)
print('WebUI ready on localhost:7860')

In [ ]:
# Cloudflare tunnel — same domain every session
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess
subprocess.Popen([
    'cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import time; time.sleep(10)
print('='*60)
print('Tunnel live. Use in Nova V2 .env:')
print(f'SD_WEBUI_URL={PUBLIC_URL}')
print('IMAGE_PROVIDER=openai,sd-webui')
print('='*60)
print('Keep this Colab tab open. Re-run when session expires.')

In [ ]:
import requests, base64
from IPython.display import Image, display
r = requests.post('http://127.0.0.1:7860/sdapi/v1/txt2img', json={
    'prompt': 'sports car, photorealistic',
    'steps': 20, 'width': 512, 'height': 512
}, timeout=120)
r.raise_for_status()
display(Image(data=base64.b64decode(r.json()['images'][0])))